# FraudShield AI — Exploratory Data Analysis

**Dataset:** Kaggle Credit Card Fraud Detection (ULB / Worldline). 284,807 transactions
from European cardholders over two days in September 2013, of which 492 are fraudulent.

Columns are `Time`, `V1`–`V28`, `Amount`, `Class`. `V1`–`V28` are PCA components released
in place of the original features for confidentiality, so we can measure how they behave
but cannot attach business meaning to any of them.

This notebook is not a gallery of plots. Each section ends with **what it means for model
design** — the decisions that flow into the training pipeline.

## Setup

If `data/raw/creditcard.csv` is absent, the notebook loads the clearly-labelled synthetic
fixture instead so it still executes. **Numbers from the fixture are meaningless** and the
banner below will tell you which source you are looking at.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "ml" / "src"))

from fraudshield.data.loader import dataset_exists, load_raw
from fraudshield.data.schema import validate_dataframe
from fraudshield.eda import run_full_eda

FIGURES = REPO / "reports" / "figures"

if dataset_exists():
    df, validation = load_raw()
    SOURCE = "KAGGLE (real)"
else:
    synthetic = REPO / "data" / "raw" / "creditcard_synthetic.csv"
    if not synthetic.is_file():
        raise SystemExit(
            "No dataset found. Run 'python ml/scripts/download_data.py' for the real data, "
            "or 'python ml/scripts/make_synthetic_data.py' for the smoke fixture."
        )
    df = pd.read_csv(synthetic)
    validation = validate_dataframe(df)
    SOURCE = "SYNTHETIC SMOKE FIXTURE - numbers below are NOT real"

print(f"Source : {SOURCE}")
print(f"Shape  : {df.shape}")
print(f"Valid  : {validation.ok}  |  warnings: {validation.warnings}")

## 1. Schema and data quality

Before anything else: does the data match the contract the pipeline is written against?

In [ ]:
print(df.dtypes.value_counts().to_string())
print()
print("Nulls          :", int(df.isna().sum().sum()))
print("Duplicate rows :", int(df.duplicated().sum()))
print("Negative amounts:", int((df.Amount < 0).sum()))
df.describe().T[["mean", "std", "min", "50%", "max"]].round(3).head(12)

**What it means for model design.** The real dataset has no nulls, which is why the
pipeline treats a null as a validation error rather than something to impute — a null here
means the upstream feed is broken, and silently filling it would hide a production
incident.

Duplicate rows do exist in the Kaggle data. We keep them: identical transactions are
plausible in real card traffic (a retried payment), and dropping them would change the base
rate that every downstream metric depends on.

In [ ]:
stats = run_full_eda(df, FIGURES)
cd = stats["class_distribution"]
print(f"Legitimate : {cd['n_legitimate']:,}")
print(f"Fraud      : {cd['n_fraud']:,}")
print(f"Fraud rate : {cd['fraud_rate'] * 100:.4f}%")
print(f"Imbalance  : 1 fraud per {cd['imbalance_ratio']:.0f} legitimate")
print()
print(f"A model that always predicts 'legitimate' scores {cd['accuracy_of_always_legit']:.4%} accuracy")
print("...while catching zero fraud.")

## 2. Class distribution — the accuracy trap

![class distribution](../../reports/figures/class_distribution.png)

**What it means for model design.** This single number drives most of the project:

- **Accuracy is disqualified as an objective.** Always predicting "legitimate" scores
  ~99.83% on the real dataset. Any metric a null model can max out is not a metric.
- **PR-AUC becomes the headline.** ROC-AUC divides by the true-negative pool, which is
  enormous here, so it looks flattering even for weak models. Precision-recall ignores
  true negatives entirely and stays sensitive to what we care about.
- **Stratified splitting is mandatory.** A random split risks landing very few positives
  in validation, making every metric noise.
- **Class imbalance needs explicit handling** — compared in section 6 of the README.

## 3. Transaction amount

![amount analysis](../../reports/figures/amount_analysis.png)

In [ ]:
amount = stats["amount"]
for key in ["legit_mean", "fraud_mean", "legit_median", "fraud_median", "legit_max", "fraud_max"]:
    print(f"{key:14s}: {amount[key]:>12,.2f}")
print(f"{'legit_skew':14s}: {amount['legit_skew']:>12,.2f}")
print(f"{'zero-amount rows':14s}: {amount['zero_amount_rows']:>12,}")

**What it means for model design.** `Amount` spans several orders of magnitude and is
heavily right-skewed, so it is `log1p`-transformed then standard-scaled before training.
Left raw, it would dominate any distance- or gradient-based model purely through its scale
while the PCA components sit near unit variance.

Amount carries some signal but is far from decisive on its own — it is a supporting feature,
not a rule. A hand-written "block transactions above X" policy would be both leaky and weak.

## 4. Time of day

![time analysis](../../reports/figures/time_analysis.png)

In [ ]:
time_stats = stats["time"]
print(f"Peak fraud-rate hour : {time_stats['peak_fraud_hour']:02d}:00 "
      f"({time_stats['peak_fraud_rate'] * 100:.3f}%)")
print(f"Lowest-volume hour   : {time_stats['lowest_volume_hour']:02d}:00")
pd.Series(time_stats["hourly_fraud_rate"]).mul(100).round(3).rename("fraud_rate_%").to_frame().T

**What it means for model design.** `Time` is *seconds elapsed since the first record*,
not a timestamp. Feeding it in raw teaches the model where a transaction sits in this
particular two-day collection window — information that does not exist for tomorrow's
traffic. That is a subtle form of leakage.

So the pipeline **drops raw `Time`** and derives a cyclical hour-of-day encoding
(`sin`/`cos`) instead. The cyclical form matters: encoded as a plain integer, hour 23 and
hour 0 would sit 23 units apart despite being one hour apart.

Volume drops sharply overnight while the fraud *rate* rises — fraud is a larger share of a
smaller pool, which is exactly the kind of interaction a tree ensemble picks up.

## 5. Which PCA components separate the classes?

![feature separation](../../reports/figures/feature_separation.png)

In [ ]:
ranked = stats["separation"]["ranked"]
ranked.head(12).round(4)

**What it means for model design.** A handful of components carry most of the
separating signal, and for the real dataset these are consistently V14, V17, V12, V10 and
V16 — with fraud shifted strongly *negative* on each.

Two consequences:

1. **Non-linear models should win.** The separation is real but overlapping; boundaries are
   not axis-aligned lines. Tree ensembles capture the interactions without manual feature
   crosses, which is why XGBoost and random forest are the leading candidates.
2. **SHAP will be interpretable at the decision level.** Since a few features dominate, a
   per-transaction attribution will usually point at a small, reviewable set — useful to a
   human analyst. But because the components are anonymised, the honest phrasing is "V14
   pushed this score up", never "the merchant category was suspicious".

## 6. Correlation structure

![correlation](../../reports/figures/correlation.png)

In [ ]:
corr = stats["correlation"]
print(f"Max  |off-diagonal correlation| : {corr['max_abs_offdiagonal']:.4f}")
print(f"Mean |off-diagonal correlation| : {corr['mean_abs_offdiagonal']:.4f}")
print()
print("Most negatively correlated with Class:")
for k, v in corr["most_negative_target_corr"].items():
    print(f"  {k:10s} {v:+.4f}")
print("Most positively correlated with Class:")
for k, v in corr["most_positive_target_corr"].items():
    print(f"  {k:10s} {v:+.4f}")

**What it means for model design.** The V components are near-orthogonal by
construction — they are PCA outputs. So:

- **No multicollinearity treatment is needed**, and no feature-selection step is warranted
  on correlation grounds.
- **Logistic regression is a fair baseline**, since its main weakness (correlated inputs)
  does not apply. If tree models beat it, the gap is genuinely about non-linearity rather
  than an artefact of the feature space.
- Correlations with `Class` are individually modest. No single feature is a fraud detector;
  the signal lives in combinations.

## 7. Outliers

Fraud detection sits close to anomaly detection, so it is worth asking whether fraudulent
transactions simply look *extreme*.

In [ ]:
outliers = stats["outliers"]
print(f"Mean extreme (|z|>3) features - legitimate : {outliers['mean_extreme_features_legit']:.3f}")
print(f"Mean extreme (|z|>3) features - fraud      : {outliers['mean_extreme_features_fraud']:.3f}")
print(f"Share of fraud with any extreme feature    : {outliers['share_fraud_with_any_extreme']:.1%}")
print(f"Share of legit with any extreme feature    : {outliers['share_legit_with_any_extreme']:.1%}")

**What it means for model design.** Fraudulent transactions are markedly more extreme
across the PCA block — which justifies the **autoencoder** as a second, unsupervised model.
Trained only on legitimate traffic, it learns what normal looks like and scores a
transaction by reconstruction error, giving a route to flagging *novel* fraud patterns that
resemble none of the 492 labelled positives.

But note the overlap: a meaningful share of legitimate transactions also contain extreme
values. Outlier-ness alone would produce far too many false alarms, which is why the
autoencoder is deployed as a novelty tripwire alongside the supervised scorer rather than
as the primary decision-maker.

**Do not clip or remove these outliers.** In most projects extreme values are noise; here
they are the signal.

## Summary — decisions carried into the pipeline

| Observation | Design decision |
|---|---|
| ~0.17% fraud | PR-AUC as the primary metric; accuracy explicitly rejected |
| Extreme imbalance | Compare class weights vs undersampling vs SMOTE, train fold only |
| Few positives | Stratified train/val/test; test opened exactly once |
| `Amount` heavily skewed | `log1p` then standard-scale |
| `Time` is a collection artefact | Drop raw `Time`; derive cyclical hour-of-day |
| Overlapping, non-linear separation | Tree ensembles as leading candidates |
| Near-orthogonal PCA components | No decorrelation step; logistic regression is a fair baseline |
| Fraud is disproportionately extreme | Autoencoder as an unsupervised novelty detector |
| Costs are asymmetric | Threshold chosen by expected cost, not left at 0.5 |
| Components are anonymised | SHAP explanations describe direction/magnitude, never business meaning |

Next: `python ml/scripts/train.py` runs the pipeline these conclusions define.